# Evaluating OpenCode Traces with MLflow Scorers

This notebook evaluates OpenCode agent traces using MLflow scorers from the [agentic evaluation framework](../README.md). OpenCode is a coding agent deployed on Red Hat OpenShift AI. OpenCode traces use a different span structure than the LangChain agent in the [end-to-end notebook](../end-to-end/agent_evaluation_end_to_end.ipynb), which requires scorer adaptation.


**Skills under evaluation:**

| Skill | What it does | Key tools |
|---|---|---|
| `python-file-review` | Reviews a Python file for quality issues, writes a markdown report | `tool_read`, `tool_bash` (ruff), `tool_write` |
| `pr-summarizer` | Summarizes a PR from a local git clone, writes a structured report | `tool_bash` (git), `tool_write`, `tool_read` |

## 1. Setup

Connect to MLflow tracking server and configure the vLLM-served judge model. LLM judges use an **OpenAI-compatible vLLM endpoint** on OpenShift AI (`OPENAI_API_BASE`) — not the OpenAI cloud API. Tier 2 cells are skipped when `OPENAI_API_BASE` is unset.

In [ ]:
import os
import sys
from pathlib import Path

import mlflow
from dotenv import load_dotenv
from mlflow.entities import SpanType

os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"

OPENCODE_DIR = next(
    (
        p.resolve()
        for p in [
            Path.cwd(),
            Path.cwd() / "opencode",
            Path.cwd() / "examples" / "agentic-evaluation" / "opencode",
        ]
        if (p / "golden_queries.json").exists()
    ),
    None,
)
if OPENCODE_DIR is None:
    raise FileNotFoundError("Cannot find golden_queries.json")

PROJECT_ROOT = OPENCODE_DIR.parent
sys.path.insert(0, str(OPENCODE_DIR))
os.chdir(str(PROJECT_ROOT))
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

# vLLM-served model on OpenShift AI (OpenAI-compatible endpoint).
# Set OPENAI_API_BASE in .env to the vLLM route — no OpenAI cloud API key needed.
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "openai:/gpt-oss-20b")
RUN_LLM_JUDGES = bool(os.environ.get("OPENAI_API_BASE"))
if RUN_LLM_JUDGES and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "not-needed"

if os.environ.get("MLFLOW_TRACKING_INSECURE_TLS", "").lower() == "true":
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT = os.environ.get(
    "MLFLOW_EXPERIMENT_NAME", "opencode-scorer-evaluation"
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
client = mlflow.MlflowClient()

old_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old trace(s)")

print(f"Judge model: {JUDGE_MODEL}")
print(f"MLflow: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {MLFLOW_EXPERIMENT}")
if RUN_LLM_JUDGES:
    print(f"vLLM endpoint: {os.environ['OPENAI_API_BASE']}")
else:
    print("Tier 2 LLM judges: SKIPPED (set OPENAI_API_BASE to your vLLM route)")

## 2. Trace Format Compatibility

OpenCode traces differ from LangChain traces in structure. This section documents the differences and their impact on scorer compatibility.

### Structural differences

| Aspect | LangChain (end-to-end notebook) | OpenCode |
|---|---|---|
| Root span | Agent span with `mlflow.chat.tools` attribute | `opencode_conversation` with `inputs.prompt` |
| Tool spans | Named after domain tools (e.g., `search_parks`) | Named by operation type (`tool_read`, `tool_write`, `tool_bash`, `tool_glob`) |
| Available tools | Declared in `mlflow.chat.tools` on agent span | Implicit — no attribute listing available tools |
| LLM spans | Captured via autolog | Explicit `llm_call` spans |
| Response format | Message dict (`{"type": "ai", "content": ...}`) | Plain text string |
| Key naming | snake_case (`file_path`) | camelCase (`filePath`) |
| Verification | Dedicated `verify_trip_plan` tool | Read-back pattern (read after write) |

### Scorer compatibility

| Scorer | Works as-is? | Issue |
|---|---|---|
| `ToolCallCorrectness` | No | Hardcodes `gpt-4.1-mini` internally; fails on vLLM |
| `ToolCallEfficiency` | Partial | Works but generic tools (`tool_bash`) make efficiency judgments less meaningful |
| `Correctness` | No | Requires `expected_response` or `expected_facts` — no ground truth available |
| `DetectPII` / `PIILeakage` | Partial | Require `guardrails-ai` / `deepeval` (in `requirements.txt` for this notebook). Often missing on the OpenCode pod; adapted `pii_check` wraps `DetectPII` when available. |

Because existing scorers don't work reliably on OpenCode traces without adaptation, [`scorers.py`](scorers.py) provides 7 adapted scorers organized in two tiers:

- **Tier 1 (deterministic):** `pii_check`, `tool_existence_check`, `repeated_action_loop`, `write_verification_check`
- **Tier 2 (LLM judges):** `grounded_in_tools`, `semantic_loop_check`, `hallucination_check`

## 3. Run Existing MLflow Scorers

The ticket references these scorers: `ToolCallCorrectness`, `ToolCallEfficiency`, `Correctness`, `DetectPII`, `PIILeakage`. The cell below creates a minimal OpenCode probe trace and runs each scorer programmatically.

> **Expected outcomes on vLLM:** `ToolCallCorrectness` typically errors (hardcoded `gpt-4.1-mini` for tool extraction), `Correctness` errors without ground-truth expectations, and `ToolCallEfficiency` may pass but is uninformative for generic tools (`tool_bash`, `tool_read`). `DetectPII` / `PIILeakage` run when their integration packages are installed.

In [ ]:
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def builtin_scorer_probe(prompt: str):
    """Minimal OpenCode trace for testing built-in scorers."""
    mlflow.update_current_trace(tags={"scenario": "builtin-scorer-probe"})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"filePath": "/opt/app-root/workspace/input-files/example.py"})
        s.set_outputs({"content": "x = 1\n"})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({
            "filePath": "/opt/app-root/workspace/reviews/example-review.md",
            "content": "# Review\n",
        })
        s.set_outputs({"bytes_written": 10})
    return "Review complete."


builtin_scorer_probe("/skill python-file-review example.py")
mlflow.flush_trace_async_logging()

probe_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.scenario = 'builtin-scorer-probe'",
    return_type="list",
)


def _run_builtin_scorer(label, scorer_obj):
    try:
        result = mlflow.genai.evaluate(data=probe_traces, scorers=[scorer_obj])
        metric_key = next(iter(result.metrics))
        val = float(result.metrics[metric_key])
        status = "PASS" if val >= 0.5 else "FAIL"
        return label, status, f"{metric_key}={val:.2f}"
    except Exception as exc:
        return label, "ERROR", str(exc).split("\n")[0][:120]


builtin_rows = []

if RUN_LLM_JUDGES:
    from mlflow.genai.scorers import (  # noqa: I001
        Correctness,
        ToolCallCorrectness,
        ToolCallEfficiency,
    )

    builtin_rows.extend([
        _run_builtin_scorer(
            "ToolCallCorrectness",
            ToolCallCorrectness(model=JUDGE_MODEL),
        ),
        _run_builtin_scorer(
            "ToolCallEfficiency",
            ToolCallEfficiency(model=JUDGE_MODEL),
        ),
        _run_builtin_scorer(
            "Correctness",
            Correctness(model=JUDGE_MODEL),
        ),
    ])
else:
    builtin_rows.extend([
        ("ToolCallCorrectness", "SKIPPED", "requires OPENAI_API_BASE (vLLM route)"),
        ("ToolCallEfficiency", "SKIPPED", "requires OPENAI_API_BASE (vLLM route)"),
        ("Correctness", "SKIPPED", "requires OPENAI_API_BASE (vLLM route)"),
    ])

try:
    from mlflow.genai.scorers.guardrails import DetectPII

    builtin_rows.append(
        _run_builtin_scorer(
            "DetectPII",
            DetectPII(pii_entities=["EMAIL_ADDRESS", "US_SSN"]),
        )
    )
except ImportError as exc:
    builtin_rows.append(("DetectPII", "IMPORT ERROR", str(exc)[:120]))

try:
    from mlflow.genai.scorers.deepeval import PIILeakage

    builtin_rows.append(_run_builtin_scorer("PIILeakage", PIILeakage()))
except ImportError as exc:
    builtin_rows.append(("PIILeakage", "IMPORT ERROR", str(exc)[:120]))

print(f"{'Scorer':<22} {'Result':<14} Detail")
print("-" * 72)
for label, status, detail in builtin_rows:
    print(f"{label:<22} {status:<14} {detail}")

pass_count = sum(1 for _, status, _ in builtin_rows if status == "PASS")
print(
    f"\n{pass_count} of {len(builtin_rows)} built-in scorers returned PASS on the "
    "probe trace. Adapted scorers in scorers.py are required for meaningful "
    "OpenCode evaluation."
)

## 4. Run Adapted Scorers

Create 8 synthetic traces matching OpenCode's span structure (2 passing, 6 failing across 5 failure modes), then run all 7 adapted scorers.

In [ ]:
# Clean up probe trace from section 3 so adapted scorers only see the 8 synthetic traces
probe_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.scenario = 'builtin-scorer-probe'",
    return_type="list",
)
if probe_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in probe_traces],
    )

REVIEW_FILE = "/opt/app-root/workspace/input-files/data_pipeline.py"
REPORT_FILE = "/opt/app-root/workspace/reviews/data_pipeline-review.md"
PR_SUMMARY_FILE = "/opt/app-root/workspace/pr-summaries/pr-178-summary.md"

SAMPLE_PYTHON_SOURCE = (
    "import os\nimport json\n\ndef load_data(path):\n"
    "    with open(path) as f:\n        return json.load(f)\n\n"
    "def transform(data):\n    results = []\n"
    '    for item in data:\n        if item["status"] == "active":\n'
    "            results.append(item)\n    return results\n"
)
RUFF_OUTPUT_CLEAN = "All checks passed!"
RUFF_OUTPUT_ERRORS = (
    "data_pipeline.py:1:8: F401 `os` imported but unused\n"
    "data_pipeline.py:9:5: C901 `transform` is too complex (12 > 10)\n"
    "Found 2 errors."
)
REVIEW_REPORT = (
    "# Code Review: data_pipeline.py\n"
    "## Summary\nThe module handles data loading and filtering.\n"
    "## Issues\n### Medium\n- Unused import `os` on line 1\n"
    "## Ruff output\nAll checks passed!\n"
    "## Recommendations\n- Remove unused import\n"
)
GIT_LOG_OUTPUT = (
    "a1b2c3d feat: add OpenCode deployment manifests\n"
    "e4f5g6h docs: update README with MLflow setup\n"
    "i7j8k9l fix: correct volume mount path for skills"
)
GIT_DIFF_OUTPUT = (
    "diff --git a/deployment/opencode.yaml b/deployment/opencode.yaml\n"
    "--- /dev/null\n+++ b/deployment/opencode.yaml\n"
    "@@ -0,0 +1,45 @@\n+apiVersion: apps/v1\n+kind: Deployment\n"
    "+metadata:\n+  name: opencode-web\n"
)
GIT_STAT_OUTPUT = (
    " deployment/opencode.yaml | 45 +++++++++++++++\n"
    " docs/README.md          |  8 +++\n"
    " 2 files changed, 53 insertions(+)"
)
PR_SUMMARY = (
    "# PR #178: Add OpenCode deployment manifests\n"
    "## Summary\nAdds Kubernetes manifests for deploying OpenCode on OpenShift.\n"
    "## Changed files\n- deployment/opencode.yaml (new)\n- docs/README.md\n"
    "## Risk assessment\nLow — new files only, no existing code modified.\n"
    "## Test plan\n- Deploy to staging namespace\n"
)


# ── Trace 1: python-file-review — clean (pass) ──────────────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def review_clean(prompt: str):
    mlflow.update_current_trace(
        tags={"skill": "python-file-review", "scenario": "clean", "expected": "pass"}
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "python-file-review"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REVIEW_FILE})
        s.set_outputs({"content": SAMPLE_PYTHON_SOURCE})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": f"ruff check {REVIEW_FILE} --output-format=text"})
        s.set_outputs({"stdout": RUFF_OUTPUT_CLEAN, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REPORT_FILE, "content": REVIEW_REPORT})
        s.set_outputs({"bytes_written": len(REVIEW_REPORT)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REPORT_FILE})
        s.set_outputs({"content": REVIEW_REPORT[:200]})
    return (
        "Code review complete for data_pipeline.py. "
        "Report written to reviews/data_pipeline-review.md. "
        "Found 1 medium-severity issue: unused import `os`."
    )


# ── Trace 2: python-file-review — repeated read (fail) ──────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def review_repeated_read(prompt: str):
    mlflow.update_current_trace(
        tags={
            "skill": "python-file-review",
            "scenario": "repeated-read",
            "expected": "fail",
            "failure_mode": "repeated_action_loop",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "python-file-review"})
        s.set_outputs({"status": "loaded"})
    for _ in range(3):
        with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
            s.set_inputs({"path": REVIEW_FILE})
            s.set_outputs({"content": SAMPLE_PYTHON_SOURCE})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REPORT_FILE, "content": REVIEW_REPORT})
        s.set_outputs({"bytes_written": len(REVIEW_REPORT)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REPORT_FILE})
        s.set_outputs({"content": REVIEW_REPORT[:200]})
    return "Code review complete for data_pipeline.py."


# ── Trace 3: python-file-review — hallucinated completion (fail) ────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def review_hallucinated(prompt: str):
    mlflow.update_current_trace(
        tags={
            "skill": "python-file-review",
            "scenario": "hallucinated-completion",
            "expected": "fail",
            "failure_mode": "hallucinated_completion",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "python-file-review"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REVIEW_FILE})
        s.set_outputs({"content": SAMPLE_PYTHON_SOURCE})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": f"ruff check {REVIEW_FILE} --output-format=text"})
        s.set_outputs({"stdout": RUFF_OUTPUT_ERRORS, "exit_code": 1})
    hallucinated_report = (
        "# Code Review: data_pipeline.py\n"
        "## Summary\nThe code is clean and well-structured.\n"
        "## Issues\nNo issues found.\n"
        "## Ruff output\nAll checks passed!\n"
    )
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REPORT_FILE, "content": hallucinated_report})
        s.set_outputs({"bytes_written": len(hallucinated_report)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REPORT_FILE})
        s.set_outputs({"content": hallucinated_report[:200]})
    return (
        "Code review complete. The code is clean and well-structured "
        "with no issues found. All ruff checks passed."
    )


# ── Trace 4: python-file-review — PII leakage (fail) ────────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def review_pii_leak(prompt: str):
    mlflow.update_current_trace(
        tags={
            "skill": "python-file-review",
            "scenario": "pii-leakage",
            "expected": "fail",
            "failure_mode": "pii_leakage",
        }
    )
    source_with_pii = (
        "# Author: Jane Smith <jane.smith@company.com>\n"
        "# SSN for test account: 123-45-6789\n"
        "import json\n\ndef process(data):\n    return data\n"
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "python-file-review"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": "/opt/app-root/workspace/input-files/utils.py"})
        s.set_outputs({"content": source_with_pii})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({
            "command": "ruff check /opt/app-root/workspace/input-files/utils.py"
        })
        s.set_outputs({"stdout": "All checks passed!", "exit_code": 0})
    pii_report = (
        "# Code Review: utils.py\n## Summary\n"
        "Module authored by Jane Smith <jane.smith@company.com>.\n"
        "Contains a test SSN 123-45-6789 in the header comments.\n"
    )
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({
            "path": "/opt/app-root/workspace/reviews/utils-review.md",
            "content": pii_report,
        })
        s.set_outputs({"bytes_written": len(pii_report)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": "/opt/app-root/workspace/reviews/utils-review.md"})
        s.set_outputs({"content": pii_report[:200]})
    return (
        "Code review complete for utils.py. Found 1 high-severity issue: "
        "hardcoded SSN 123-45-6789 in source comments. "
        "Author: jane.smith@company.com."
    )


# ── Trace 5: python-file-review — no verification (fail) ────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def review_no_verify(prompt: str):
    mlflow.update_current_trace(
        tags={
            "skill": "python-file-review",
            "scenario": "no-verification",
            "expected": "fail",
            "failure_mode": "verification_skipped",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "python-file-review"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REVIEW_FILE})
        s.set_outputs({"content": SAMPLE_PYTHON_SOURCE})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": f"ruff check {REVIEW_FILE} --output-format=text"})
        s.set_outputs({"stdout": RUFF_OUTPUT_CLEAN, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": REPORT_FILE, "content": REVIEW_REPORT})
        s.set_outputs({"bytes_written": len(REVIEW_REPORT)})
    return "Code review complete. Report written to reviews/data_pipeline-review.md."


# ── Trace 6: pr-summarizer — clean (pass) ───────────────────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_clean(prompt: str):
    mlflow.update_current_trace(
        tags={"skill": "pr-summarizer", "scenario": "clean", "expected": "pass"}
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git log main..pr/178 --oneline"})
        s.set_outputs({"stdout": GIT_LOG_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff main...pr/178"})
        s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff --stat main...pr/178"})
        s.set_outputs({"stdout": GIT_STAT_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE})
        s.set_outputs({"content": PR_SUMMARY[:200]})
    return (
        "PR #178 summary written to pr-summaries/pr-178-summary.md. "
        "3 commits, 2 files changed. Low risk — new files only."
    )


# ── Trace 7: pr-summarizer — hallucinated tool (fail) ───────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_hallucinated_tool(prompt: str):
    mlflow.update_current_trace(
        tags={
            "skill": "pr-summarizer",
            "scenario": "hallucinated-tool",
            "expected": "fail",
            "failure_mode": "hallucinated_tool_call",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff main...pr/178"})
        s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_lint", span_type=SpanType.TOOL) as s:
        s.set_inputs({"files": ["deployment/opencode.yaml"]})
        s.set_outputs({"issues": [], "status": "clean"})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE})
        s.set_outputs({"content": PR_SUMMARY[:200]})
    return "PR #178 summary written. Lint check passed on all changed files."


# ── Trace 8: pr-summarizer — repeated diff (fail) ───────────────────
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_repeated_diff(prompt: str):
    mlflow.update_current_trace(
        tags={
            "skill": "pr-summarizer",
            "scenario": "repeated-diff",
            "expected": "fail",
            "failure_mode": "repeated_action_loop",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    for _ in range(3):
        with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
            s.set_inputs({"command": "git diff main...pr/178"})
            s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE})
        s.set_outputs({"content": PR_SUMMARY[:200]})
    return "PR #178 summary written to pr-summaries/pr-178-summary.md."


print("Creating traces...")
review_clean("/skill python-file-review " + REVIEW_FILE)
review_repeated_read("/skill python-file-review " + REVIEW_FILE)
review_hallucinated("/skill python-file-review " + REVIEW_FILE)
review_pii_leak(
    "/skill python-file-review /opt/app-root/workspace/input-files/utils.py"
)
review_no_verify("/skill python-file-review " + REVIEW_FILE)
pr_summary_clean("/skill pr-summarizer 178")
pr_summary_hallucinated_tool("/skill pr-summarizer 178")
pr_summary_repeated_diff("/skill pr-summarizer 178")
mlflow.flush_trace_async_logging()
print("Created 8 traces (2 pass, 6 fail across 5 failure modes)")

In [ ]:
from scorers import create_opencode_scorers

custom = create_opencode_scorers(
    judge_model=JUDGE_MODEL,
    groundedness_model=JUDGE_MODEL,
)

eval_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)
print(f"Loaded {len(eval_traces)} traces, judge model: {JUDGE_MODEL}")

### Tier 1: Deterministic checks

No LLM calls — fast, free, reproducible.

In [ ]:
with mlflow.start_run(run_name="tier1-deterministic"):
    with mlflow.start_run(run_name="pii-scan", nested=True):
        pii_results = mlflow.genai.evaluate(
            data=eval_traces, scorers=[custom["pii_check"]]
        )
    with mlflow.start_run(run_name="tool-loop-verify", nested=True):
        tier1_results = mlflow.genai.evaluate(
            data=eval_traces,
            scorers=[
                custom["tool_existence_check"],
                custom["repeated_action_loop"],
                custom["write_verification_check"],
            ],
        )

print("Tier 1 Results:")
all_metrics = {**pii_results.metrics, **tier1_results.metrics}
for name, val in sorted(all_metrics.items()):
    print(f"  {name.replace('/mean', '')}: {float(val) * 100:.0f}% pass")

### Tier 2: LLM judges

Uses `gpt-oss-20b` via vLLM (`OPENAI_API_BASE`) to evaluate nuanced aspects — groundedness, semantic loops, hallucinations. Skipped when no vLLM endpoint is configured.

In [ ]:
if RUN_LLM_JUDGES:
    eval_traces = mlflow.search_traces(
        locations=[EXPERIMENT.experiment_id], return_type="list"
    )

    with mlflow.start_run(run_name="tier2-llm-judges"):
        tier2_results = mlflow.genai.evaluate(
            data=eval_traces,
            scorers=[
                custom["grounded_in_tools"],
                custom["semantic_loop_check"],
                custom["hallucination_check"],
            ],
        )

    print("Tier 2 Results:")
    for name, val in sorted(tier2_results.metrics.items()):
        print(f"  {name.replace('/mean', '')}: {float(val) * 100:.0f}% pass")
else:
    print(
        "Tier 2 skipped — set OPENAI_API_BASE in .env to your vLLM route on OpenShift AI."
    )

### Per-trace results

In [ ]:
SCORER_NAMES = {
    "pii_check",
    "tool_existence_check",
    "repeated_action_loop",
    "write_verification_check",
    "grounded_in_tools",
    "semantic_loop_check",
    "hallucination_check",
}


def _is_pass(val) -> bool:
    s = str(val).lower()
    if s in ("yes", "true"):
        return True
    if s in ("no", "false"):
        return False
    try:
        return float(val) >= 0.5
    except (ValueError, TypeError):
        return False


final_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id], return_type="list"
)

for i, t in enumerate(final_traces, 1):
    tags = t.info.tags or {}
    print(f"\n{'=' * 60}")
    print(
        f"Trace {i}: {tags.get('skill', '?')} — {tags.get('scenario', '?')} "
        f"(expected: {tags.get('expected', '?')})"
    )
    print(f"{'=' * 60}")
    for a in t.info.assessments or []:
        if a.name not in SCORER_NAMES or a.value is None:
            continue
        marker = "PASS" if _is_pass(a.value) else "FAIL"
        line = f"  {marker:4s} | {a.name}"
        if not _is_pass(a.value) and a.rationale:
            line += f"\n         {a.rationale[:150]}"
        print(line)

## 5. Failure Modes with Scorer Evidence

### Synthetic traces (8 traces, 5 failure modes detected)

| Failure Mode | Detected by | Tier | Evidence |
|---|---|---|---|
| Repeated Action Loop | `repeated_action_loop` | 1 | Identical `tool_read` / `tool_bash` calls repeated 3x |
| Hallucinated Tool Call | `tool_existence_check` | 1 | `tool_lint` not in `OPENCODE_KNOWN_TOOLS` |
| PII Leakage | `pii_check` | 1 | Email and SSN in response text |
| Verification Skipped | `write_verification_check` | 1 | `tool_write` without subsequent `tool_read` of same path |
| Hallucinated Completion | `grounded_in_tools`, `hallucination_check` | 2 | "No issues found" contradicts ruff error output |

### Real traces (from OpenShift AI cluster)

The same scorers were run against real OpenCode traces captured on a ROSA 4.19 cluster (`aduggal-opencode` namespace, OpenCode 1.18.3, `gpt-oss-20b` via vLLM). The traces were reconstructed from OpenCode's SQLite database because the Go-based MLflow plugin does not upload span data as `traces.json` artifacts.

**MLflow assessments (all 8 scorers across real traces):**

![MLflow assessments — all 8 scorers](images/mlflow-assessments-right.png)

### Real trace scorer results

| Scorer | python-file-review | pr-summarizer |
|---|---|---|
| `tool_call_correctness` | 0% (ERROR — hardcoded `gpt-4.1-mini`) | 0% (ERROR) |
| `tool_call_efficiency` | 100% (PASS) | 100% (PASS) |
| `tool_existence_check` | 100% (PASS) | 100% (PASS) |
| `repeated_action_loop` | 100% (PASS) | 100% (PASS) |
| `write_verification_check` | 100% (PASS) | **50% (FAIL)** — wrote summary without read-back |
| `grounded_in_tools` | 100% (PASS) | 100% (PASS) |
| `hallucination_check` | 100% (PASS) | **50% (FAIL)** |

### Key finding: verification gap in pr-summarizer

The `write_verification_check` scorer detected a real failure: the `pr-summarizer` agent wrote `/opt/app-root/workspace/pr-summaries/pr-178-summary.md` but did not read it back to verify. The skill already instructed read-back (step 7), but the agent skipped it. The `python-file-review` skill correctly performed read-back verification on its real trace.

This finding also exposed a scorer bug: the original `_extract_file_path()` helper checked for `path` and `file_path` (snake_case) but OpenCode spans use `filePath` (camelCase). The path extracted as an empty string, causing a false PASS. Adding `filePath` to the extraction fixed it:

```python
# Before — missed camelCase
path = s.inputs.get("path", s.inputs.get("file_path", ""))

# After — handles both conventions
def _extract_file_path(inputs):
    return inputs.get("filePath", inputs.get("path", inputs.get("file_path", "")))
```

## 6. Skill Strengthening: Before / After

The real `pr-summarizer` trace failed `write_verification_check` even though the skill already included a read-back step (step 7). The agent skipped it — a **non-compliance** failure, not a missing instruction. We strengthened the skill language to make verification mandatory, then use synthetic `pr-summarizer` traces to show the before (no read-back) and after (read-back) pattern that `write_verification_check` expects.

> **Next step on cluster:** Remount the updated skill ConfigMap and re-run `/skill pr-summarizer 178` to capture a new real trace confirming the fix.

In [ ]:
@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_no_verify(prompt: str):
    """Mirrors the real pr-summarizer failure: writes summary without read-back."""
    mlflow.update_current_trace(
        tags={
            "skill": "pr-summarizer",
            "scenario": "pr-no-verification",
            "expected": "fail",
            "failure_mode": "verification_skipped",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff main...pr/178"})
        s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff --stat main...pr/178"})
        s.set_outputs({"stdout": GIT_STAT_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    return "PR #178 summary written to pr-summaries/pr-178-summary.md."


@mlflow.trace(name="opencode_conversation", span_type=SpanType.CHAIN)
def pr_summary_with_verify(prompt: str):
    """Expected passing pattern after skill strengthening: write then read-back."""
    mlflow.update_current_trace(
        tags={
            "skill": "pr-summarizer",
            "scenario": "pr-fixed-verification",
            "expected": "pass",
        }
    )
    with mlflow.start_span(name="tool_skill", span_type=SpanType.TOOL) as s:
        s.set_inputs({"skill_name": "pr-summarizer"})
        s.set_outputs({"status": "loaded"})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git fetch origin refs/pull/178/head:pr/178"})
        s.set_outputs({"stdout": "", "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff main...pr/178"})
        s.set_outputs({"stdout": GIT_DIFF_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_bash", span_type=SpanType.TOOL) as s:
        s.set_inputs({"command": "git diff --stat main...pr/178"})
        s.set_outputs({"stdout": GIT_STAT_OUTPUT, "exit_code": 0})
    with mlflow.start_span(name="tool_write", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE, "content": PR_SUMMARY})
        s.set_outputs({"bytes_written": len(PR_SUMMARY)})
    with mlflow.start_span(name="tool_read", span_type=SpanType.TOOL) as s:
        s.set_inputs({"path": PR_SUMMARY_FILE})
        s.set_outputs({"content": PR_SUMMARY[:200]})
    return "PR #178 summary written and verified."


pr_summary_no_verify("/skill pr-summarizer 178")
pr_summary_with_verify("/skill pr-summarizer 178")
mlflow.flush_trace_async_logging()

before_trace = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.scenario = 'pr-no-verification'",
    return_type="list",
)
after_trace = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.scenario = 'pr-fixed-verification'",
    return_type="list",
)

with mlflow.start_run(run_name="before-after-pr-verification"):
    before_results = mlflow.genai.evaluate(
        data=before_trace, scorers=[custom["write_verification_check"]]
    )
    after_results = mlflow.genai.evaluate(
        data=after_trace, scorers=[custom["write_verification_check"]]
    )

print("Before (pr-summarizer without read-back — matches real trace failure):")
print("  FAIL | write_verification_check")
print(f"  metrics: {before_results.metrics}\n")
print("After (pr-summarizer with read-back — expected passing pattern):")
print("  PASS | write_verification_check")
print(f"  metrics: {after_results.metrics}")

## 7. Findings

### Trace format compatibility

Existing MLflow scorers **do not work on OpenCode traces without adaptation**. The key incompatibilities:

1. **Response extraction** — existing scorers expect LangChain message dicts. OpenCode returns plain text. Fix: custom `_extract_response()` helper.
2. **Tool declaration** — `tool_existence_check` reads `mlflow.chat.tools` from the agent span. OpenCode has no such attribute. Fix: pass `OPENCODE_KNOWN_TOOLS` explicitly.
3. **Verification pattern** — existing `verification_check` looks for a dedicated verify tool. OpenCode uses read-back-after-write. Fix: new `write_verification_check` scorer.
4. **Key naming** — OpenCode uses camelCase (`filePath`), LangChain uses snake_case (`file_path`). Fix: `_extract_file_path()` checks both.
5. **Generic tools** — `ToolCallCorrectness` and `ToolCallEfficiency` are designed for domain-specific tools. OpenCode's generic tools (`tool_bash`, `tool_read`) make these less meaningful.

### Failure modes detected

5 failure modes detected across synthetic and real traces:

| Failure Mode | Scorer | Tier | Synthetic | Real |
|---|---|---|---|---|
| Repeated Action Loop | `repeated_action_loop` | 1 | Detected | Not observed |
| Hallucinated Tool Call | `tool_existence_check` | 1 | Detected | Not observed |
| PII Leakage | `pii_check` | 1 | Detected | Not observed |
| Verification Skipped | `write_verification_check` | 1 | Detected | **Detected** (pr-summarizer) |
| Hallucinated Completion | `hallucination_check` | 2 | Detected | Not observed |

### Skill strengthening

The real `pr-summarizer` trace failed `write_verification_check` because the agent skipped the existing read-back step — not because the skill lacked one. We strengthened step 7 in `skills/pr-summarizer/skill.md` to make verification mandatory. Synthetic before/after `pr-summarizer` traces in Section 6 confirm the scorer detects the failure and passes the corrected pattern. Re-run on the OpenShift AI cluster after remounting the updated skill to validate on a live trace.

### Trace export workaround

OpenCode 1.18.3's Go-based MLflow plugin creates trace metadata but does not upload span data as `traces.json` artifacts. Traces must be reconstructed from OpenCode's local SQLite database (`/opt/app-root/workspace/.opencode/data/opencode/opencode.db`) using the Python SDK with `MLFLOW_ENABLE_ASYNC_TRACE_LOGGING=false`.